In [11]:
# ============================================================
# CELL 1: Install real dependencies
# ============================================================
%%capture
!pip install -U firecrawl-anydoc docling pymupdf pydantic-settings

In [31]:
%%capture
!pip install -U langchain-text-splitters

In [33]:
%%capture
!pip install -U groq

In [105]:
# ============================================================
# CELL 2: Logging
# ============================================================
import logging
import sys

def setup_logger(name: str = "rag_harness") -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger

logger = setup_logger()
logger.info("Logger initialized")

08:16:59 | INFO     | rag_harness | Logger initialized


INFO:rag_harness:Logger initialized


In [ ]:
# ============================================================
# CELL 3: Settings
# ============================================================
from pydantic_settings import BaseSettings
from functools import lru_cache


class Settings(BaseSettings):
    # ---- LLM (for contextual chunking preambles) ----
    MODEL_NAME: str = 
    API_URL: str = 
    AUTH_TOKEN: str = 
    TIMEOUT_SECONDS: int = 60
    MAX_RETRIES: int = 3
    RETRY_DELAY_SECONDS: int = 5

    # ---- PDF page classification thresholds (real, structural) ----
    PRIMARY_CONFIDENCE_THRESHOLD: float = 0.85
    IMAGE_COVERAGE_SCAN_THRESHOLD: float = 0.85   # >85% page covered by image -> likely scan
    MIN_TEXT_CHARS_PER_PAGE: int = 40             # below this, text layer is probably absent/broken

    # ---- Post-extraction quality gate ----
    MIN_EXTRACTED_CHARS: int = 20
    MAX_REPLACEMENT_CHAR_RATIO: float = 0.05
    MAX_JUNK_CHAR_RATIO: float = 0.3

    # ---- Chunking ----
    CHUNK_SIZE_CHARS: int = 1000
    CHUNK_OVERLAP_CHARS: int = 150
    WHOLE_DOC_CONTEXT_THRESHOLD_CHARS: int = 20000

    class Config:
        env_file = ".env"
        extra = "ignore"


@lru_cache
def get_settings() -> Settings:
    logger.info("Settings loaded")
    return Settings()


settings = get_settings()

08:16:59 | INFO     | rag_harness | Settings loaded


/tmp/ipykernel_2468/3289838099.py:8: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class Settings(BaseSettings):
INFO:rag_harness:Settings loaded


In [107]:
# ============================================================
# CELL 4: Exceptions
# ============================================================

class HarnessError(Exception):
    """Base class for all harness-level errors."""


class LLMClientError(HarnessError):
    """Raised when the LLM backend fails after all retries."""


class IngestionError(HarnessError):
    """Raised when a document fails to parse/route/extract during ingestion."""

In [ ]:
# ============================================================
# CELL 5: LLM Client (mymodel) — used for contextual chunking preambles
# ============================================================
import requests
import time
from abc import ABC, abstractmethod


class BaseLLMClient(ABC):
    @abstractmethod
    def generate(self, system_prompt: str, user_content: str, max_tokens: int = 500, temperature: float = 0.0) -> str:
        ...


class myLLMClient(BaseLLMClient):
    def __init__(self, settings: Settings):
        self.settings = settings
        self.logger = logging.getLogger("rag_harness.llm.mymodel")

    def generate(self, system_prompt: str, user_content: str, max_tokens: int = 500, temperature: float = 0.0) -> str:
        payload = {
            "model": self.settings.MODEL_NAME,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
            ],
            "temperature": temperature,
            "max_tokens": max_tokens,
        }
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.settings.AUTH_TOKEN}",
        }

        last_error = None
        for attempt in range(1, self.settings.MAX_RETRIES + 1):
            try:
                self.logger.info(f"Calling mymodel (attempt {attempt}/{self.settings.MAX_RETRIES})")
                resp = requests.post(
                    self.settings.API_URL, headers=headers, json=payload,
                    timeout=self.settings.TIMEOUT_SECONDS,
                )
                if resp.status_code >= 400:
                    last_error = f"http_{resp.status_code}: {resp.text[:1000]}"
                    self.logger.warning(last_error)
                    if 400 <= resp.status_code < 500:
                        break
                    if attempt < self.settings.MAX_RETRIES:
                        time.sleep(self.settings.RETRY_DELAY_SECONDS)
                    continue
                resp.raise_for_status()
                result = resp.json()
                return result["choices"][0]["message"]["content"]
            except requests.exceptions.RequestException as e:
                last_error = f"request_failed: {str(e)}"
                self.logger.warning(last_error)
                if attempt < self.settings.MAX_RETRIES:
                    time.sleep(self.settings.RETRY_DELAY_SECONDS)
            except (KeyError, IndexError) as e:
                last_error = f"unexpected_response_shape: {str(e)}"
                self.logger.error(last_error)
                if attempt < self.settings.MAX_RETRIES:
                    time.sleep(self.settings.RETRY_DELAY_SECONDS)

        raise LLMClientError(f"myLLMClient giving up after {self.settings.MAX_RETRIES} attempts: {last_error}")


class LLMClientFactory:
    _registry = {"mymodel": myLLMClient}

    @classmethod
    def create(cls, backend: str, settings: Settings) -> BaseLLMClient:
        if backend not in cls._registry:
            raise ValueError(f"Unknown backend '{backend}'")
        logger.info(f"Instantiating LLM backend: {backend}")
        return cls._registry[backend](settings)


llm_client: BaseLLMClient = LLMClientFactory.create("mymodel", settings)

In [ ]:
# ============================================================
# CELL 5b: Groq LLM client with retry -> model-switch fallback logic
# ============================================================


from groq import Groq
from groq import RateLimitError, APIStatusError
import time


# class GroqSettings(BaseSettings):
#     GROQ_API_KEY: str =
#     GROQ_PRIMARY_MODEL: str = "openai/gpt-oss-20b"
#     GROQ_FALLBACK_MODEL: str = "qwen/qwen3.8-27b"
#     GROQ_MAX_RETRIES_PER_MODEL: int = 2
#     GROQ_RETRY_DELAY_SECONDS: int = 3
#     GROQ_TIMEOUT_SECONDS: int = 30

#     class Config:
#         env_file = ".env"
#         extra = "ignore"
class GroqSettings(BaseSettings):
    GROQ_API_KEY: str = 
    GROQ_PRIMARY_MODEL: str = "qwen/qwen3.8-27b"
    GROQ_FALLBACK_MODEL: str = "openai/gpt-oss-20b"
    GROQ_MAX_RETRIES_PER_MODEL: int = 2
    GROQ_RETRY_DELAY_SECONDS: int = 3
    GROQ_TIMEOUT_SECONDS: int = 30

    class Config:
        env_file = ".env"
        extra = "ignore"

groq_settings = GroqSettings()




/tmp/ipykernel_2468/1263583839.py:22: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class GroqSettings(BaseSettings):


In [110]:
# ============================================================
# CELL 5b (updated): GroqLLMClient — minimize reasoning ONLY when the
# fallback (a reasoning model) actually gets used
# ============================================================
class GroqLLMClient(BaseLLMClient):
    def __init__(self, api_key: str, model: str, timeout: int):
        self.model = model
        self.timeout = timeout
        self.client = Groq(api_key=api_key)
        self.logger = logging.getLogger(f"rag_harness.llm.groq.{model}")

    def generate(self, system_prompt: str, user_content: str, max_tokens: int = 500, temperature: float = 0.0) -> str:
        try:
            kwargs = dict(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content},
                ],
                max_tokens=max_tokens,
                temperature=temperature,
                timeout=self.timeout,
            )
            # Only gpt-oss (reasoning model) needs this — Qwen doesn't reason by default
            if "gpt-oss" in self.model:
                kwargs["reasoning_effort"] = "low"

            response = self.client.chat.completions.create(**kwargs)
            content = response.choices[0].message.content

            if not content or not content.strip():
                finish_reason = response.choices[0].finish_reason
                self.logger.error(
                    f"Empty content from model={self.model}, finish_reason={finish_reason}"
                )
                raise LLMClientError(f"Empty content (finish_reason={finish_reason})")

            return content
        except (RateLimitError, APIStatusError):
            raise
        except LLMClientError:
            raise
        except Exception as e:
            self.logger.error(f"Unexpected error calling model={self.model}: {e}")
            raise
class GroqFallbackClient(BaseLLMClient):
    """Retries the primary model up to N times; if all retries fail specifically
    due to rate limiting, switches to the fallback model for that call.
    Non-rate-limit errors also count toward exhausting primary's retries,
    since a broken primary should still hand off rather than hard-fail."""

    def __init__(self, settings: GroqSettings):
        self.settings = settings
        self.logger = logging.getLogger("rag_harness.llm.groq_fallback")
        self.primary = GroqLLMClient(settings.GROQ_API_KEY, settings.GROQ_PRIMARY_MODEL, settings.GROQ_TIMEOUT_SECONDS)
        self.fallback = GroqLLMClient(settings.GROQ_API_KEY, settings.GROQ_FALLBACK_MODEL, settings.GROQ_TIMEOUT_SECONDS)

    def generate(self, system_prompt: str, user_content: str, max_tokens: int = 500, temperature: float = 0.0) -> str:
        last_error = None

        # Try primary model up to GROQ_MAX_RETRIES_PER_MODEL times
        for attempt in range(1, self.settings.GROQ_MAX_RETRIES_PER_MODEL + 1):
            try:
                self.logger.info(f"Primary model attempt {attempt}/{self.settings.GROQ_MAX_RETRIES_PER_MODEL}")
                return self.primary.generate(system_prompt, user_content, max_tokens, temperature)
            except RateLimitError as e:
                last_error = f"rate_limit: {e}"
                self.logger.warning(f"Primary rate-limited (attempt {attempt}) — will retry or fall back")
                if attempt < self.settings.GROQ_MAX_RETRIES_PER_MODEL:
                    time.sleep(self.settings.GROQ_RETRY_DELAY_SECONDS)
            except Exception as e:
                last_error = f"error: {e}"
                self.logger.warning(f"Primary failed (attempt {attempt}): {e}")
                if attempt < self.settings.GROQ_MAX_RETRIES_PER_MODEL:
                    time.sleep(self.settings.GROQ_RETRY_DELAY_SECONDS)

        # Primary exhausted -> switch to fallback model
        self.logger.warning(f"Primary model exhausted after {self.settings.GROQ_MAX_RETRIES_PER_MODEL} attempts "
                             f"(last_error={last_error}) — switching to fallback model={self.settings.GROQ_FALLBACK_MODEL}")
        try:
            return self.fallback.generate(system_prompt, user_content, max_tokens, temperature)
        except Exception as e:
            self.logger.error(f"Fallback model also failed: {e}")
            raise LLMClientError(
                f"Both primary ({self.settings.GROQ_PRIMARY_MODEL}) and "
                f"fallback ({self.settings.GROQ_FALLBACK_MODEL}) failed. "
                f"Primary last error: {last_error}. Fallback error: {e}"
            )


groq_fallback_client: BaseLLMClient = GroqFallbackClient(groq_settings)

In [111]:
# ============================================================
# CELL 6: Data models
# ============================================================
from pydantic import BaseModel, Field
from typing import Optional, Literal
from enum import Enum
import uuid


class RouteDecision(str, Enum):
    ANYDOC = "anydoc"
    DOCLING = "docling"


class PageClassification(BaseModel):
    page_num: int
    route: RouteDecision
    text_char_count: int
    image_coverage_ratio: float
    has_text_layer: bool
    reason: str


class ChunkMetadata(BaseModel):
    doc_id: str
    doc_version: str
    source_type: Literal["pdf", "md"]
    page_num: Optional[int] = None
    ingestion_route: Optional[str] = None


class Chunk(BaseModel):
    chunk_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    raw_text: str
    contextual_text: str
    metadata: ChunkMetadata

In [115]:
# ============================================================
# CELL 7d (simplified): Strategy selector collapses to TWO outcomes only —
# mixed pages now route to Docling instead of REGION_SPLIT
# ============================================================
class PDFPageStrategy(str, Enum):
    WHOLE_PAGE_ANYDOC = "whole_page_anydoc"    # uniform clean text page
    WHOLE_PAGE_DOCLING = "whole_page_docling"  # scanned OR mixed (table+scan+text)


class PageStrategyDecision(BaseModel):
    page_num: int
    strategy: PDFPageStrategy
    reason: str


class PDFPageStrategySelector:
    """Two-way decision only. Any page that isn't cleanly uniform text
    (i.e. has meaningful image coverage OR a detected table) goes entirely
    to Docling — no region splitting."""

    def __init__(self, settings: Settings):
        self.settings = settings
        self.logger = logging.getLogger("rag_harness.ingestion.strategy_selector")

    def select_strategy(self, page: fitz.Page) -> PageStrategyDecision:
        text = page.get_text("text")
        text_char_count = len(text.strip())
        has_text_layer = text_char_count >= self.settings.MIN_TEXT_CHARS_PER_PAGE

        image_coverage = self._compute_image_coverage(page)
        has_table = self._has_table(page)

        # Uniform clean text: real text, negligible image coverage, no table
        if has_text_layer and image_coverage < 0.15 and not has_table:
            return PageStrategyDecision(
                page_num=page.number, strategy=PDFPageStrategy.WHOLE_PAGE_ANYDOC,
                reason="uniform_text_page",
            )

        # Everything else — fully scanned, or mixed (text+table+scan) — goes to Docling
        if not has_text_layer and image_coverage >= self.settings.IMAGE_COVERAGE_SCAN_THRESHOLD:
            reason = "uniform_scan_page"
        elif has_table:
            reason = "mixed_page_has_table"
        else:
            reason = "mixed_signals_text_and_image"

        return PageStrategyDecision(page_num=page.number, strategy=PDFPageStrategy.WHOLE_PAGE_DOCLING, reason=reason)

    def _compute_image_coverage(self, page: fitz.Page) -> float:
        page_area = page.rect.width * page.rect.height
        if page_area == 0:
            return 0.0
        image_area = 0.0
        for img in page.get_images(full=True):
            xref = img[0]
            try:
                for r in page.get_image_rects(xref):
                    image_area += r.width * r.height
            except Exception:
                continue
        return min(image_area / page_area, 1.0)

    def _has_table(self, page: fitz.Page) -> bool:
        try:
            tables = page.find_tables()
            return len(tables.tables) > 0
        except Exception as e:
            self.logger.warning(f"Table detection failed on page {page.number}: {e}")
            return False


page_strategy_selector = PDFPageStrategySelector(settings)

In [116]:
# ============================================================
# CELL 8: Post-extraction quality gate — catches garbled-font failures
# on AnyDoc's output, real string checks
# ============================================================
class ExtractionQualityChecker:
    def __init__(self, settings: Settings):
        self.settings = settings
        self.logger = logging.getLogger("rag_harness.ingestion.quality_check")

    def is_extraction_suspicious(self, extracted_text: str) -> tuple[bool, str]:
        stripped = extracted_text.strip()
        if len(stripped) < self.settings.MIN_EXTRACTED_CHARS:
            return True, "extracted_text_too_short"

        replacement_count = extracted_text.count("\ufffd")
        ratio = replacement_count / max(len(extracted_text), 1)
        if ratio > self.settings.MAX_REPLACEMENT_CHAR_RATIO:
            return True, f"high_replacement_char_ratio_{ratio:.3f}"

        junk_chars = sum(1 for c in extracted_text if not (c.isalnum() or c.isspace() or c in ".,;:!?()-'\""))
        junk_ratio = junk_chars / max(len(extracted_text), 1)
        if junk_ratio > self.settings.MAX_JUNK_CHAR_RATIO:
            return True, f"high_junk_char_ratio_{junk_ratio:.3f}"

        return False, ""


extraction_quality_checker = ExtractionQualityChecker(settings)

In [118]:
# ============================================================
# CELL 9b (updated): routing log signature needs the rerouted/reason kwargs
# ============================================================
class IngestionRoutingLog:
    def __init__(self):
        self.logger = logging.getLogger("rag_harness.ingestion.routing_log")
        self.records: list[dict] = []

    def log_strategy(self, doc_id: str, strategy: PageStrategyDecision, rerouted: bool = False, reroute_reason: str = None):
        record = {
            "doc_id": doc_id,
            "page_num": strategy.page_num,
            "strategy": strategy.strategy.value,
            "reason": strategy.reason,
            "rerouted": rerouted,
            "reroute_reason": reroute_reason,
        }
        self.records.append(record)
        self.logger.info(f"STRATEGY doc={doc_id} page={strategy.page_num} -> {record['strategy']} ({record['reason']})")


routing_log = IngestionRoutingLog()

In [119]:
# ============================================================
# CELL 10 (simplified): FileRouter — back to clean two-way routing,
# batched Docling per document, AnyDoc using real PyMuPDF text
# (no more fabricated "txt" format call — that was the earlier bug)
# ============================================================
class FileRouter:
    def __init__(
        self,
        strategy_selector: PDFPageStrategySelector,
        quality_checker: ExtractionQualityChecker,
        routing_log: IngestionRoutingLog,
    ):
        self.strategy_selector = strategy_selector
        self.quality_checker = quality_checker
        self.routing_log = routing_log
        self.logger = logging.getLogger("rag_harness.ingestion.file_router")
        self._docling_converter = DocumentConverter()

    def ingest_file(self, file_path: str, doc_id: str, doc_version: str) -> list[dict]:
        if file_path.lower().endswith(".pdf"):
            return self._ingest_pdf(file_path, doc_id, doc_version)
        elif file_path.lower().endswith(".md"):
            return self._ingest_md(file_path, doc_id, doc_version)
        else:
            raise IngestionError(f"Unsupported file type for {file_path} — only PDF and MD supported currently")

    def _ingest_pdf(self, file_path: str, doc_id: str, doc_version: str) -> list[dict]:
        try:
            pdf_doc = fitz.open(file_path)
        except Exception as e:
            raise IngestionError(f"PDF open failed: {e}")

        strategies = []
        try:
            for page in pdf_doc:
                strategies.append(self.strategy_selector.select_strategy(page))
        except Exception as e:
            pdf_doc.close()
            raise IngestionError(f"Strategy selection failed: {e}")

        docling_needed = any(s.strategy == PDFPageStrategy.WHOLE_PAGE_DOCLING for s in strategies)
        docling_batch_texts = {}
        if docling_needed:
            docling_batch_texts = self._extract_whole_pages_with_docling(file_path, doc_id)

        results = []
        for strategy in strategies:
            page = pdf_doc[strategy.page_num]
            self.logger.info(f"doc={doc_id} page={strategy.page_num}: strategy={strategy.strategy.value} ({strategy.reason})")

            if strategy.strategy == PDFPageStrategy.WHOLE_PAGE_ANYDOC:
                text = page.get_text("text")  # real structural extraction, no fabricated format call
                suspicious, reason = self.quality_checker.is_extraction_suspicious(text)
                if suspicious:
                    self.logger.warning(f"Page {strategy.page_num}: extraction suspicious ({reason}) — rerouting to Docling")
                    if not docling_batch_texts:
                        docling_batch_texts = self._extract_whole_pages_with_docling(file_path, doc_id)
                    text = docling_batch_texts.get(strategy.page_num, "")
                    self.routing_log.log_strategy(doc_id, strategy, rerouted=True, reroute_reason=reason)
                    results.append({"page_num": strategy.page_num, "text": text, "route": "docling_fallback"})
                    continue

                self.routing_log.log_strategy(doc_id, strategy)
                results.append({"page_num": strategy.page_num, "text": text, "route": "anydoc"})

            else:  # WHOLE_PAGE_DOCLING — covers both fully-scanned AND mixed pages
                text = docling_batch_texts.get(strategy.page_num, "")
                self.routing_log.log_strategy(doc_id, strategy)
                results.append({"page_num": strategy.page_num, "text": text, "route": "docling"})

        pdf_doc.close()
        return results

    def _extract_whole_pages_with_docling(self, file_path: str, doc_id: str) -> dict[int, str]:
        self.logger.info(f"doc={doc_id}: batched Docling conversion (scanned + mixed pages)")
        try:
            result = self._docling_converter.convert(file_path)
            doc = result.document

            page_texts: dict[int, list[str]] = {}
            for item, _level in doc.iterate_items():
                if not hasattr(item, "prov") or not item.prov:
                    continue
                for prov in item.prov:
                    page_no = prov.page_no - 1  # Docling 1-indexed -> pipeline 0-indexed
                    page_texts.setdefault(page_no, [])
                    if hasattr(item, "text") and item.text:
                        page_texts[page_no].append(item.text)
                    elif hasattr(item, "export_to_markdown"):
                        try:
                            page_texts[page_no].append(item.export_to_markdown(doc))
                        except Exception:
                            page_texts[page_no].append(str(item))

            result_dict = {page_no: "\n\n".join(parts) for page_no, parts in page_texts.items()}

            if not result_dict:
                self.logger.warning(f"doc={doc_id}: per-page grouping produced nothing — falling back to full-doc markdown")
                full_markdown = doc.export_to_markdown()
                result_dict = {p_no - 1: full_markdown for p_no in doc.pages.keys()}

            self.logger.info(f"doc={doc_id}: Docling batch conversion complete, {len(result_dict)} pages extracted")
            return result_dict
        except Exception as e:
            self.logger.error(f"Batched Docling extraction failed for doc={doc_id}: {e}")
            raise IngestionError(f"Docling batch extraction failed: {e}")

    def _ingest_md(self, file_path: str, doc_id: str, doc_version: str) -> list[dict]:
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
        except Exception as e:
            raise IngestionError(f"MD file read failed: {e}")
        return [{"page_num": None, "text": content, "route": "direct_md"}]


file_router = FileRouter(page_strategy_selector, extraction_quality_checker, routing_log)

In [121]:
# ============================================================
# CELL 11a: Structure-aware chunk splitting via LangChain's
# RecursiveCharacterTextSplitter — respects markdown structure and
# sentence boundaries instead of raw character slicing
# ============================================================


from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter


class StructureAwareChunker:
    """Two-stage split:
    1. Split on markdown headers first, so a chunk never crosses a section boundary
       (a '## Section X' heading always starts a new logical unit).
    2. Within each header-defined section, recursively split on paragraph -> sentence
       -> word -> char, in that priority order, only falling back to a hard char cut
       when nothing else fits the size budget."""

    def __init__(self, settings: Settings):
        self.settings = settings
        self.logger = logging.getLogger("rag_harness.chunking.splitter")

        self.header_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")],
            strip_headers=False,  # keep header text IN the chunk — needed for context
        )

        self.recursive_splitter = RecursiveCharacterTextSplitter(
            chunk_size=settings.CHUNK_SIZE_CHARS,
            chunk_overlap=settings.CHUNK_OVERLAP_CHARS,
            separators=["\n\n", "\n", ". ", " ", ""],  # tries these in order, biggest-unit-first
            length_function=len,
        )

    def split(self, text: str) -> list[str]:
        try:
            header_sections = self.header_splitter.split_text(text)
        except Exception as e:
            self.logger.warning(f"Markdown header split failed ({e}) — falling back to plain recursive split")
            header_sections = None

        final_chunks = []
        if header_sections:
            for section in header_sections:
                section_text = section.page_content
                if len(section_text) <= self.settings.CHUNK_SIZE_CHARS:
                    final_chunks.append(section_text)
                else:
                    final_chunks.extend(self.recursive_splitter.split_text(section_text))
        else:
            final_chunks = self.recursive_splitter.split_text(text)

        final_chunks = [c for c in final_chunks if c.strip()]
        self.logger.info(f"Structure-aware split produced {len(final_chunks)} chunks")
        return final_chunks


structure_aware_chunker = StructureAwareChunker(settings)

In [122]:
# ============================================================
# CELL 11b (updated): ContextualChunker with concurrent preamble generation
# ============================================================
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import textwrap

class ContextualChunker:
    def __init__(self, llm_client: BaseLLMClient, settings: Settings, max_workers: int = 8):
        self.llm_client = llm_client
        self.settings = settings
        self.max_workers = max_workers
        self.logger = logging.getLogger("rag_harness.chunking")
        self._log_lock = threading.Lock()  # avoid interleaved log lines from multiple threads

    def split_into_base_chunks(self, text: str) -> list[str]:
        return structure_aware_chunker.split(text)

    def generate_context_preamble(self, full_document_text: str, target_chunk: str, chunk_index: int, all_chunks: list[str]) -> str:
        use_whole_doc = len(full_document_text) <= self.settings.WHOLE_DOC_CONTEXT_THRESHOLD_CHARS

        if use_whole_doc:
            context_source = full_document_text
        else:
            prev_chunk = all_chunks[chunk_index - 1] if chunk_index > 0 else ""
            next_chunk = all_chunks[chunk_index + 1] if chunk_index < len(all_chunks) - 1 else ""
            context_source = f"...{prev_chunk}\n[TARGET CHUNK HERE]\n{next_chunk}..."

        system_prompt = "You situate a text chunk within its document. Be concise: 1-2 sentences only."
        user_content = textwrap.dedent(f"""
            Document context:
            {context_source}

            Target chunk:
            {target_chunk}

            Give a short 1-2 sentence context to situate this chunk within the overall document.
            Do not repeat the chunk text. Do not include document names, page numbers, or IDs —
            those are tracked separately as metadata.
        """).strip()

        try:
            preamble = self.llm_client.generate(system_prompt, user_content, max_tokens=300, temperature=0.0)
            if not preamble or not preamble.strip():
                with self._log_lock:
                    self.logger.error(f"Chunk {chunk_index}: LLM returned empty preamble")
                return ""
            return preamble.strip()
        except Exception as e:
            with self._log_lock:
                self.logger.error(f"Chunk {chunk_index}: preamble generation raised {type(e).__name__}: {e}")
            return ""

    def _build_single_chunk(self, idx: int, raw_chunk: str, page_entry: dict, full_document_text: str, all_chunks: list[str], doc_id: str, doc_version: str, source_type: str) -> Chunk:
        """Runs inside a worker thread — builds ONE chunk end to end."""
        preamble = self.generate_context_preamble(full_document_text, raw_chunk, idx, all_chunks)
        contextual_text = f"{preamble}\n{raw_chunk}" if preamble else raw_chunk

        with self._log_lock:
            self.logger.info(f"Chunk {idx+1}/{len(all_chunks)} built (doc={doc_id}, page={page_entry.get('page_num')})")

        return Chunk(
            raw_text=raw_chunk,
            contextual_text=contextual_text,
            metadata=ChunkMetadata(
                doc_id=doc_id,
                doc_version=doc_version,
                source_type=source_type,
                page_num=page_entry.get("page_num"),
                ingestion_route=page_entry.get("route"),
            ),
        )

    def chunk_document(self, doc_id: str, doc_version: str, source_type: str, page_texts: list[dict]) -> list[Chunk]:
        full_document_text = "\n\n".join(p["text"] for p in page_texts)
        all_chunks_flat = []
        chunk_page_map = []

        for page_entry in page_texts:
            base_chunks = self.split_into_base_chunks(page_entry["text"])
            for bc in base_chunks:
                all_chunks_flat.append(bc)
                chunk_page_map.append(page_entry)

        # Preserve chunk ORDER even though work happens concurrently —
        # submit with index, collect results into a pre-sized list, not append-as-completed
        result_chunks: list[Optional[Chunk]] = [None] * len(all_chunks_flat)

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            future_to_idx = {
                executor.submit(
                    self._build_single_chunk,
                    idx, raw_chunk, chunk_page_map[idx], full_document_text, all_chunks_flat, doc_id, doc_version, source_type,
                ): idx
                for idx, raw_chunk in enumerate(all_chunks_flat)
            }

            for future in as_completed(future_to_idx):
                idx = future_to_idx[future]
                try:
                    result_chunks[idx] = future.result()
                except Exception as e:
                    self.logger.error(f"Chunk {idx} failed entirely: {e} — building without preamble as last resort")
                    result_chunks[idx] = Chunk(
                        raw_text=all_chunks_flat[idx],
                        contextual_text=all_chunks_flat[idx],
                        metadata=ChunkMetadata(
                            doc_id=doc_id, doc_version=doc_version, source_type=source_type,
                            page_num=chunk_page_map[idx].get("page_num"),
                            ingestion_route=chunk_page_map[idx].get("route"),
                        ),
                    )

        self.logger.info(f"Chunked doc={doc_id} into {len(result_chunks)} contextual chunks (concurrent, workers={self.max_workers})")
        return result_chunks


contextual_chunker = ContextualChunker(groq_fallback_client, settings, max_workers=3)

In [124]:
# ============================================================
# CELL 12: Full ingestion pipeline entry point
# ============================================================
class IngestionPipeline:
    def __init__(self, file_router: FileRouter, chunker: ContextualChunker):
        self.file_router = file_router
        self.chunker = chunker
        self.logger = logging.getLogger("rag_harness.ingestion.pipeline")

    def ingest(self, file_path: str, doc_id: str, doc_version: str) -> list[Chunk]:
        self.logger.info(f"=== Ingesting {file_path} (doc_id={doc_id}, version={doc_version}) ===")
        source_type = "pdf" if file_path.lower().endswith(".pdf") else "md"
        try:
            page_texts = self.file_router.ingest_file(file_path, doc_id, doc_version)
            chunks = self.chunker.chunk_document(doc_id, doc_version, source_type, page_texts)
            self.logger.info(f"=== Ingestion complete: {len(chunks)} chunks produced for {doc_id} ===")
            return chunks
        except HarnessError:
            raise
        except Exception as e:
            self.logger.error(f"Ingestion pipeline failed unexpectedly: {e}")
            raise IngestionError(f"Ingestion failed for {file_path}: {e}")


ingestion_pipeline = IngestionPipeline(file_router, contextual_chunker)

In [125]:
# ============================================================
# CELL 13: Test it — upload a real PDF or MD file in Colab first
# ============================================================
# from google.colab import files
# uploaded = files.upload()   # pick a .pdf or .md file
file_path = "/content/sample_document_merged.pdf"

chunks = ingestion_pipeline.ingest(file_path, doc_id="test_doc_1", doc_version="v1")
# for c in chunks[:3]:
#     print("---")
#     print("Route:", c.metadata.ingestion_route, "| Page:", c.metadata.page_num)
#     # print("Contextual text:", c.contextual_text[:200])

print(f"\nTotal chunks: {len(chunks)}")
print(f"\nRouting log entries: {len(routing_log.records)}")
for r in routing_log.records:
    print(r)

08:17:05 | INFO     | rag_harness.ingestion.pipeline | === Ingesting /content/sample_document_merged.pdf (doc_id=test_doc_1, version=v1) ===


INFO:rag_harness.ingestion.pipeline:=== Ingesting /content/sample_document_merged.pdf (doc_id=test_doc_1, version=v1) ===


Consider using the pymupdf_layout package for a greatly improved page layout analysis.
08:17:05 | INFO     | rag_harness.ingestion.file_router | doc=test_doc_1: batched Docling conversion (scanned + mixed pages)


INFO:rag_harness.ingestion.file_router:doc=test_doc_1: batched Docling conversion (scanned + mixed pages)
[INFO] 2026-09-15 08:17:05,386 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-15 08:17:05,388 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-15 08:17:05,399 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-15 08:17:05,401 [RapidOCR] main.py:50: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-15 08:17:05,537 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-15 08:17:05,538 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-15 08:17:05,540 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-09-15 08:17:05,543 [RapidOCR] main.py:50: Using /usr/local/lib/python3.13/di

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

...

08:20:47 | INFO     | rag_harness.chunking | Chunked doc=test_doc_1 into 12 contextual chunks (concurrent, workers=3)


INFO:rag_harness.chunking:Chunked doc=test_doc_1 into 12 contextual chunks (concurrent, workers=3)


08:20:47 | INFO     | rag_harness.ingestion.pipeline | === Ingestion complete: 12 chunks produced for test_doc_1 ===


INFO:rag_harness.ingestion.pipeline:=== Ingestion complete: 12 chunks produced for test_doc_1 ===



Total chunks: 12

Routing log entries: 5
{'doc_id': 'test_doc_1', 'page_num': 0, 'strategy': 'whole_page_anydoc', 'reason': 'uniform_text_page', 'rerouted': False, 'reroute_reason': None}
{'doc_id': 'test_doc_1', 'page_num': 1, 'strategy': 'whole_page_docling', 'reason': 'uniform_scan_page', 'rerouted': False, 'reroute_reason': None}
{'doc_id': 'test_doc_1', 'page_num': 2, 'strategy': 'whole_page_docling', 'reason': 'mixed_page_has_table', 'rerouted': False, 'reroute_reason': None}
{'doc_id': 'test_doc_1', 'page_num': 3, 'strategy': 'whole_page_docling', 'reason': 'mixed_page_has_table', 'rerouted': False, 'reroute_reason': None}
{'doc_id': 'test_doc_1', 'page_num': 4, 'strategy': 'whole_page_docling', 'reason': 'mixed_signals_text_and_image', 'rerouted': False, 'reroute_reason': None}


In [126]:
len(chunks)

12

In [29]:
for ch in chunks[:3]:
  print(ch)

chunk_id='53005fd4-5055-4e15-a563-745ffaae6988' raw_text="<!-- image -->\n\n## Dear Commissioner:\n\nThere is no higher priority for the U.S. Environmental Protection Agency than protecting public health and ensuring the safety of our nation's drinking water. Under the Safe Drinking Water Act (SDWA), «State» and other states have the primary responsibility for the implementation and enforcement of drinking water regulations, while the EPA is tasked with oversight of state efforts. Recent events in Flint, Michigan, and other U.S. cities, have led to important discussions about the safety of our nation's drinking water supplies. I am writing today to ask you to join in taking action to strengthen our safe drinking water programs, consistent with our shared recognition of the critical importance of safe drinking water for the health of all Americans.\n\nFirst, with most states having primacy under SDWA, we need to work together to ensure that states are taking action to demonstrate that t

In [127]:
chunks[0]

Chunk(chunk_id='bb22e29d-874d-4362-9801-e717fb911ea2', raw_text='1\nSample PDF  \nCreated for testing PDFObject  \nThis PDF is three pages long. Three long pages. Or three short pages if\nyou’re optimistic. Is it the same as saying “three long minutes”, knowing\nthat all minutes are the same duration, and one cannot possibly be longer\nthan the other? If these pages are all the same size, can one possibly be\nlonger than the other?  \nI digress. Here’s some Latin. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Integer nec\nodio. Praesent libero. Sed cursus ante dapibus diam. Sed nisi. Nulla quis sem at nibh elementum\nimperdiet. Duis sagittis ipsum. Praesent mauris. Fusce nec tellus sed augue semper porta. Mauris\nmassa. Vestibulum lacinia arcu eget nulla. Class aptent taciti sociosqu ad litora torquent per\nconubia nostra, per inceptos himenaeos. Curabitur sodales ligula in libero.  \nSed dignissim lacinia nunc. Curabitur tortor. Pellentesque nibh. Aenean quam. In scelerisqu

In [128]:
chunks[1]

Chunk(chunk_id='f89bcad0-6013-4168-be95-3a069b478145', raw_text='Sed dignissim lacinia nunc. Curabitur tortor. Pellentesque nibh. Aenean quam. In scelerisque sem\nat dolor. Maecenas mattis. Sed convallis tristique sem. Proin ut ligula vel nunc egestas porttitor.\nMorbi lectus risus, iaculis vel, suscipit quis, luctus non, massa. Fusce ac turpis quis ligula lacinia\naliquet. Mauris ipsum. Nulla metus metus, ullamcorper vel, tincidunt sed, euismod in, nibh.  \nQuisque volutpat condimentum velit. Class aptent taciti sociosqu ad litora torquent per conubia\nnostra, per inceptos himenaeos. Nam nec ante. Sed lacinia, urna non tincidunt mattis, tortor neque\nadipiscing diam, a cursus ipsum ante quis turpis. Nulla facilisi. Ut fringilla. Suspendisse potenti.\nNunc feugiat mi a tellus consequat imperdiet. Vestibulum sapien. Proin quam. Etiam ultrices.  \nSuspendisse in justo eu magna luctus suscipit. Sed lectus. Integer euismod lacus luctus magna.\nQuisque cursus, metus vitae pharetra auctor, s

In [129]:
chunks[2]

Chunk(chunk_id='397fa76b-2c50-49d9-b54d-428a2ac0dc10', raw_text='Quisque cursus, metus vitae pharetra auctor, sem massa mattis sem, at interdum magna augue\neget diam. Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere cubilia Curae;\nMorbi lacinia molestie dui. Praesent blandit dolor. Sed non quam. In vel mi sit amet augue congue\nelementum. Morbi in ipsum sit amet pede facilisis laoreet. Donec lacus nunc, viverra nec, blandit\nvel, egestas et, augue. Vestibulum tincidunt malesuada tellus. Ut ultrices ultrices enim. Curabitur\nsit amet mauris.  \nMorbi in dui quis est pulvinar ullamcorper. Nulla facilisi. Integer lacinia sollicitudin massa. Cras\nmetus. Sed aliquet risus a tortor. Integer id quam. Morbi mi. Quisque nisl felis, venenatis tristique,\ndignissim in, ultrices sit amet, augue. Proin sodales libero eget ante. Nulla quam. Aenean laoreet.\nVestibulum nisi lectus, commodo ac, facilisis ac, ultricies eu, pede. Ut orci risus, accumsan\nporttitor, cursus quis

In [56]:
# ============================================================
# CELL: Diagnostic — call the Groq client directly, once, to see the real error
# ============================================================
try:
    test_response = groq_fallback_client.generate(
        system_prompt="You situate a text chunk within its document. Be concise: 1-2 sentences only.",
        user_content="Document context:\nThis is a test document about drinking water safety.\n\nTarget chunk:\nThe EPA is writing to state commissioners.\n\nGive a short 1-2 sentence context to situate this chunk.",
        max_tokens=80,
        temperature=0.0,
    )
    print("SUCCESS:", test_response)
except Exception as e:
    print("FAILED WITH:", type(e).__name__, "-", str(e))

07:33:37 | INFO     | rag_harness.llm.groq_fallback | Primary model attempt 1/2


INFO:rag_harness.llm.groq_fallback:Primary model attempt 1/2


SUCCESS: 


In [57]:
test_response

''

In [79]:
# ============================================================
# Diagnostic: inspect Docling's actual page structure
# ============================================================
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
result = converter.convert("/content/sample_document_merged.pdf")
doc = result.document

print("Type of doc.pages:", type(doc.pages))
print("Keys:", list(doc.pages.keys())[:5])

first_key = list(doc.pages.keys())[0]
first_page_item = doc.pages[first_key]
print("\nType of a page item:", type(first_page_item))
print("Page item attributes:", [a for a in dir(first_page_item) if not a.startswith("_")])

[INFO] 2026-09-15 07:59:25,671 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-15 07:59:25,672 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-15 07:59:25,689 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-15 07:59:25,690 [RapidOCR] main.py:50: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-15 07:59:25,982 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-15 07:59:25,984 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-15 07:59:25,986 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-09-15 07:59:25,988 [RapidOCR] main.py:50: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-09-15 07:59:26,285 [RapidOCR] 

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[WARNING] 2026-09-15 08:00:21,400 [RapidOCR] main.py:132: The text detection result is empty


Type of doc.pages: <class 'dict'>
Keys: [1, 2, 3, 4, 5]

Type of a page item: <class 'docling_core.types.doc.common.reference.PageItem'>
Page item attributes: ['construct', 'copy', 'dict', 'from_orm', 'image', 'json', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_validate', 'model_validate_json', 'model_validate_strings', 'page_no', 'parse_file', 'parse_obj', 'parse_raw', 'schema', 'schema_json', 'size', 'update_forward_refs', 'validate']


In [80]:
# ============================================================
# Diagnostic 2: confirm doc.iterate_items() + item.prov actually works
# ============================================================
count = 0
sample_items = []
for item, level in doc.iterate_items():
    count += 1
    if count <= 5:
        sample_items.append({
            "type": type(item).__name__,
            "has_text": hasattr(item, "text"),
            "text_preview": getattr(item, "text", None),
            "has_prov": hasattr(item, "prov"),
            "prov": getattr(item, "prov", None),
        })

print(f"Total items: {count}")
for s in sample_items:
    print(s)


Total items: 29
{'type': 'SectionHeaderItem', 'has_text': True, 'text_preview': 'Sample PDF', 'has_prov': True, 'prov': [ProvenanceItem(page_no=1, bbox=BoundingBox(l=190.3089, t=734.504, r=432.303, b=702.0719432048681, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 10))]}
{'type': 'SectionHeaderItem', 'has_text': True, 'text_preview': 'Created for testing PDFObject', 'has_prov': True, 'prov': [ProvenanceItem(page_no=1, bbox=BoundingBox(l=179.4676, t=681.362, r=437.8163, b=665.145971602434, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 29))]}
{'type': 'TextItem', 'has_text': True, 'text_preview': "This PDF is three pages long. Three long pages. Or three short pages if you're optimistic. Is it the same as saying 'three long minutes', knowing that all minutes are the same duration, and one cannot possibly be longer than the other? If these pages are all the same size, can one possibly be longer than the other?", 'has_prov': True, 'prov': [Provenanc